# Session 9 · RAG from scratch

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai chromadb
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)
import chromadb

# 1. INDEX (as in Session 8)
docs = [(f"{name[:-3]}-{n}", p.strip()) for name, text in lumo.handbook()
        for n, p in enumerate(x for x in text.split("\n\n")[1:] if x.strip())]
kb = chromadb.Client().get_or_create_collection("handbook")
kb.add(ids=[i for i, _ in docs], documents=[t for _, t in docs])

SYSTEM = """You answer customer questions for Lumo using ONLY the numbered sources.
Cite the source id in square brackets after each fact, e.g. [05-returns-0].
If the sources don't contain the answer, say "I don't know based on our handbook" and suggest contacting support."""

def rag(question, k=3):
    hits = kb.query(query_texts=[question], n_results=k)                             # 2. RETRIEVE
    sources = "\n".join(f"[{i}] {t}" for i, t in zip(hits["ids"][0], hits["documents"][0]))
    prompt = f"<sources>\n{sources}\n</sources>\n\nQuestion: {question}"         # 3. AUGMENT
    return llm.ask(prompt, system=SYSTEM).text, hits["ids"][0]                        # 4. GENERATE

for q in ["My lamp arrived broken, do I need to send it back?",
          "How long is the warranty on a built-in LED module?",
          "Do you offer gift wrapping?"]:                # not in the handbook: watch it say so
    answer, used = rag(q)
    print(f"Q: {q}\nRetrieved: {used}\nA: {answer}\n")